# LeetCode 856 — Score of Parentheses bằng Python

Link đề: https://leetcode.com/problems/score-of-parentheses/ · Độ khó: **Medium** · Chủ đề: Stack, String, Parentheses

Skeleton LeetCode:

```python
class Solution:
    def scoreOfParentheses(self, s: str) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên). Điểm cốt lõi: mỗi cặp `"()"` **trực tiếp** (lá) đóng góp `2^(độ sâu)` vào tổng, trong đó độ sâu là số cặp ngoặc bao quanh nó. Notebook dùng oracle **sinh chuỗi từ chính văn phạm của đề** (`()`, `AB`, `(A)`) để đối chiếu mọi cách.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tính điểm của chuỗi ngoặc cân bằng theo ba luật `()` = 1, `AB` = `A + B`, `(A)` = `2 * A`.
2. Nắm công cụ Python: duyệt chuỗi, **list làm stack**, đệ quy trên chỉ số, `enumerate`, `2 ** depth` hoặc dịch bit `1 << depth`.
3. Đi từ **định nghĩa đệ quy (tách chuỗi con theo số dư ngoặc)** → **stack lưu điểm** → **đếm độ sâu (`2^d` cho mỗi `"()"`)** → **bộ phân tích đệ quy (recursive descent) theo chỉ số**.
4. Hiểu mấu chốt: điểm là **tổng các lá**, mỗi lá `"()"` đóng góp `2^d` với `d` là độ sâu của nó (vì mỗi cặp ngoặc bao ngoài nhân đôi).
5. Tránh lỗi kinh điển: coi `"(())"` là `1 + 1`, cộng thay vì nhân khi gặp ngoặc đóng, quên trường hợp `"()"` rỗng bên trong, nhầm độ sâu (đếm cả chính nó), và `2 ** d` sai khi cập nhật độ sâu trước/sau.

## 2. Đề bài & ví dụ

Cho chuỗi ngoặc **cân bằng** `s`. Trả về **điểm** của chuỗi theo luật:

- `"()"` có điểm `1`.
- `AB` có điểm `A + B` (`A`, `B` là chuỗi cân bằng).
- `(A)` có điểm `2 * A`.

**Ví dụ 1:** `s = "()"` → `1`.

**Ví dụ 2:** `s = "(())"` → `2` (`2 * 1`).

**Ví dụ 3:** `s = "()()"` → `2` (`1 + 1`).

**Ràng buộc:** `2 <= s.length <= 50`; `s` chỉ gồm `'('`, `')'`; `s` cân bằng.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [("()", 1), ("(())", 2), ("()()", 2)]
for s, want in examples:
    print(f"s = {s!r} -> {want}")

s = '()' -> 1
s = '(())' -> 2
s = '()()' -> 2


## 3. Kiến thức nền

### 3.1 Chuỗi ngoặc cân bằng và độ sâu

Chuỗi cân bằng khi (i) mọi tiền tố có số `'('` ≥ số `')'` và (ii) toàn chuỗi hai số bằng nhau. Đặt `bal` = số `'('` trừ số `')'` khi duyệt: mỗi `'('` tăng `bal` thêm 1, mỗi `')'` giảm 1. `bal` chính là **độ sâu** hiện tại (số cặp ngoặc đang mở).

In [2]:
def depths(s):
    out, bal = [], 0
    for ch in s:
        bal += 1 if ch == "(" else -1
        out.append(bal)
    return out


s = "(()(()))"
print(s)
print(depths(s), "-> cuối bằng 0 và không âm:", depths(s)[-1] == 0 and min(depths(s)) >= 0)

(()(()))
[1, 2, 1, 2, 3, 2, 1, 0] -> cuối bằng 0 và không âm: True


### 3.2 Phân rã nguyên thể (primitive decomposition)

Mọi chuỗi cân bằng khác rỗng **tách duy nhất** thành các khối nguyên thể `(A1)(A2)…` — mỗi khối là một cặp ngoặc ngoài cùng bao một chuỗi cân bằng (có thể rỗng). Điểm cả chuỗi là tổng điểm các khối; điểm khối `(A)` là `1` nếu `A` rỗng, ngược lại `2 * score(A)`. Khối nguyên thể kết thúc ở chỗ `bal` quay về `0`.

In [3]:
def primitives(s):
    out, bal, start = [], 0, 0
    for i, ch in enumerate(s):
        bal += 1 if ch == "(" else -1
        if bal == 0:
            out.append(s[start:i + 1])
            start = i + 1
    return out


print(primitives("(()(()))()(())"))

['(()(()))', '()', '(())']


### 3.3 List làm stack

`stack.append(x)` đẩy, `stack.pop()` lấy phần tử cuối, `stack[-1]` xem đỉnh. Cả ba là `O(1)`. Ta dùng stack để lưu "điểm đang cộng dồn trong mỗi cặp ngoặc đang mở".

### 3.4 Luỹ thừa 2

`2 ** d` hoặc `1 << d`. Với `n ≤ 50` độ sâu tối đa `25` nên `2^25` vẫn nhỏ; Python không tràn, nhưng ở C++ `int` cũng đủ (`≤ 2^24`).

In [4]:
print([1 << d for d in range(6)], [2 ** d for d in range(6)])
print("độ sâu tối đa với n = 50 là", 50 // 2, "-> điểm tối đa", 1 << (50 // 2 - 1))

[1, 2, 4, 8, 16, 32] [1, 2, 4, 8, 16, 32]
độ sâu tối đa với n = 50 là 25 -> điểm tối đa 16777216


## 4. Ý tưởng nền tảng / trực giác

Vẽ `"(()(()))"` bằng cây: gốc là một cặp ngoài cùng; bên trong có hai con: `"()"` và `"(())"`.

```
( ( ) ( ( ) ) )
```

- Ngoài cùng nhân 2.
- Bên trong: `"()"` (=1) cộng `"(())"` (=2·1=2) → 3.
- Tổng: `2 · 3 = 6`.

Nhìn theo **lá**: có hai lá `"()"`. Lá thứ nhất nằm trong 1 cặp ngoài (độ sâu 1) nên đóng góp `2^1 = 2`; lá thứ hai nằm trong 2 cặp (độ sâu 2) nên đóng góp `2^2 = 4`. Tổng `2 + 4 = 6` ✓.

Lý do: mỗi cặp ngoặc bao ngoài nhân **toàn bộ** điểm bên trong với 2, nên lá ở độ sâu `d` được nhân `2` đúng `d` lần.

In [5]:
s = "(()(()))"
bal, total = 0, 0
for i, ch in enumerate(s):
    if ch == "(":
        bal += 1
    else:
        if s[i - 1] == "(":
            print(f"lá '()' tại vị trí {i-1}..{i}: độ sâu (số cặp bao ngoài) = {bal - 1} -> đóng góp {2 ** (bal - 1)}")
            total += 2 ** (bal - 1)
        bal -= 1
print("tổng =", total)

lá '()' tại vị trí 1..2: độ sâu (số cặp bao ngoài) = 1 -> đóng góp 2
lá '()' tại vị trí 4..5: độ sâu (số cặp bao ngoài) = 2 -> đóng góp 4
tổng = 6


## 5. Cách 1 — Brute Force: đệ quy theo định nghĩa

**Ý tưởng:** áp thẳng định nghĩa. Với chuỗi `s`: tìm các khối nguyên thể (chỗ `bal` về `0`). Với mỗi khối `(A)`: nếu `A` rỗng điểm `1`, ngược lại `2 * score(A)`. Cộng các khối.

In [6]:
def score_recursive(s):
    if s == "":
        return 0
    total, bal, start = 0, 0, 0
    for i, ch in enumerate(s):
        bal += 1 if ch == "(" else -1
        if bal == 0:
            inner = s[start + 1:i]                       # bỏ cặp ngoặc ngoài
            total += 1 if inner == "" else 2 * score_recursive(inner)
            start = i + 1
    return total


for s, want in examples:
    assert score_recursive(s) == want
print([score_recursive(s) for s, _ in examples], score_recursive("(()(()))"))

[1, 2, 2] 6


**Độ phức tạp:** mỗi tầng đệ quy quét lại chuỗi con, và cắt chuỗi tốn `O(n)`; chuỗi lồng sâu `((((...))))` gây `O(n²)` thời gian, `O(n)` chiều sâu đệ quy. Với `n ≤ 50` vẫn nhanh nhưng lãng phí.

**Nút thắt:** ta cắt và quét lại chuỗi con nhiều lần. Chỉ cần **một lượt** nếu biết điểm cộng dồn của từng cặp đang mở.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai góc nhìn một lượt:

1. **Stack điểm:** mỗi cặp ngoặc đang mở giữ "điểm đã tích luỹ bên trong nó". Gặp `'('` đẩy `0` mới. Gặp `')'` lấy `inner = stack.pop()`, điểm của cặp này là `1` nếu `inner == 0` (rỗng) ngược lại `2 * inner`, rồi **cộng vào đỉnh stack** (cặp cha).
2. **Độ sâu:** mỗi lần gặp `"()"` liền nhau, cộng `2^(độ sâu hiện tại)` vào kết quả — không cần stack.

Cả hai cho `O(n)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** stack điểm, **Cách 3** đếm độ sâu (`2^d`), **Cách 4** bộ phân tích đệ quy theo chỉ số (không cắt chuỗi).

### Cách 2 — Stack điểm

**Giải thích từng đoạn:**

1. `stack = [0]`: phần tử đáy là điểm cấp ngoài cùng.
2. Gặp `'('`: `stack.append(0)` mở cặp mới với điểm tích luỹ `0`.
3. Gặp `')'`: `inner = stack.pop()`; `stack[-1] += max(1, 2 * inner)` (nếu `inner = 0` thì cặp rỗng `()` được `1`, ngược lại `2*inner`; `max(1, 2*inner)` làm cả hai).
4. Kết quả là `stack[0]`.

In [7]:
def score_stack(s):
    stack = [0]
    for ch in s:
        if ch == "(":
            stack.append(0)
        else:
            inner = stack.pop()
            stack[-1] += max(1, 2 * inner)               # () = 1, (A) = 2A
    return stack[0]


for s, want in examples:
    assert score_stack(s) == want
print([score_stack(s) for s, _ in examples], score_stack("(()(()))"))

[1, 2, 2] 6


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionStack:
    def scoreOfParentheses(self, s: str) -> int:
        stack = [0]
        for ch in s:
            if ch == "(":
                stack.append(0)
            else:
                inner = stack.pop()
                stack[-1] += max(1, 2 * inner)
        return stack[0]

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (độ sâu tối đa `n/2`).

### Cách 3 — Đếm độ sâu: mỗi `"()"` đóng góp `2^d`

**Giải thích từng đoạn:**

1. Giữ `depth`. Gặp `'('`: `depth += 1`. Gặp `')'`: `depth -= 1`.
2. Nếu ký tự trước đó là `'('` (tức vừa đóng một cặp rỗng `"()"`), cộng `1 << depth` vào kết quả (sau khi đã giảm `depth`, `depth` chính là số cặp bao ngoài).
3. Không cần stack.

In [9]:
def score_depth(s):
    res = depth = 0
    for i, ch in enumerate(s):
        if ch == "(":
            depth += 1
        else:
            depth -= 1
            if s[i - 1] == "(":                          # vừa đóng một lá "()"
                res += 1 << depth
    return res


for s, want in examples:
    assert score_depth(s) == want
print([score_depth(s) for s, _ in examples], score_depth("(()(()))"))

[1, 2, 2] 6


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionDepth:
    def scoreOfParentheses(self, s: str) -> int:
        res = depth = 0
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            else:
                depth -= 1
                if s[i - 1] == "(":
                    res += 1 << depth
        return res

**Độ phức tạp:** `O(n)` thời gian, **`O(1)` bộ nhớ**. Đây là cách tối ưu về không gian.

### Cách 4 — Bộ phân tích đệ quy theo chỉ số

**Ý tưởng:** viết đúng văn phạm `S → ε | (S) S`... mà không cắt chuỗi: hàm `parse(i)` đọc một dãy khối từ vị trí `i` tới khi gặp `')'` hoặc hết chuỗi, trả `(điểm, vị trí tiếp)`. Mỗi khối `'(' parse ')'` cho `1` nếu bên trong trống, ngược lại `2 * điểm`.

In [11]:
def score_parser(s):
    n = len(s)

    def parse(i):
        total = 0
        while i < n and s[i] == "(":
            inner, i = parse(i + 1)                      # đọc nội dung bên trong cặp
            i += 1                                       # bỏ ')' tương ứng
            total += 1 if inner == 0 else 2 * inner
        return total, i

    return parse(0)[0]


for s, want in examples:
    assert score_parser(s) == want
print([score_parser(s) for s, _ in examples], score_parser("(()(()))"))

[1, 2, 2] 6


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionParser:
    def scoreOfParentheses(self, s: str) -> int:
        n = len(s)

        def parse(i: int):
            total = 0
            while i < n and s[i] == "(":
                inner, i = parse(i + 1)
                i += 1
                total += 1 if inner == 0 else 2 * inner
            return total, i

        return parse(0)[0]

**Độ phức tạp:** `O(n)` thời gian (mỗi ký tự thăm một lần), `O(n)` ngăn xếp đệ quy. Với `n ≤ 50` an toàn.

## 8. TRACE từng bước

**Trace Cách 2 (stack)** với `s = "(()(()))"`:

| ký tự | thao tác | stack sau |
|---|---|---|
| `(` | đẩy 0 | `[0, 0]` |
| `(` | đẩy 0 | `[0, 0, 0]` |
| `)` | pop 0 → `max(1,0)=1` cộng vào đỉnh | `[0, 1]` |
| `(` | đẩy 0 | `[0, 1, 0]` |
| `(` | đẩy 0 | `[0, 1, 0, 0]` |
| `)` | pop 0 → +1 | `[0, 1, 1]` |
| `)` | pop 1 → `+2` | `[0, 3]` |
| `)` | pop 3 → `+6` | `[6]` |

Kết quả `6` ✓. Hàm trace in từng bước:

In [13]:
def trace_stack(s):
    stack = [0]
    print(f"s = {s!r}")
    print(f"{'i':>2} {'ch':>3} {'thao tác':<28} {'stack':<18}")
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(0)
            act = "đẩy 0"
        else:
            inner = stack.pop()
            add = max(1, 2 * inner)
            stack[-1] += add
            act = f"pop {inner} -> cộng {add} vào đỉnh"
        print(f"{i:>2} {ch:>3} {act:<28} {stack}")
    print("=>", stack[0])
    return stack[0]


assert trace_stack("(()(()))") == 6

s = '(()(()))'
 i  ch thao tác                     stack             
 0   ( đẩy 0                        [0, 0]
 1   ( đẩy 0                        [0, 0, 0]
 2   ) pop 0 -> cộng 1 vào đỉnh     [0, 1]
 3   ( đẩy 0                        [0, 1, 0]
 4   ( đẩy 0                        [0, 1, 0, 0]
 5   ) pop 0 -> cộng 1 vào đỉnh     [0, 1, 1]
 6   ) pop 1 -> cộng 2 vào đỉnh     [0, 3]
 7   ) pop 3 -> cộng 6 vào đỉnh     [6]
=> 6


**Trace Cách 3 (độ sâu)** với cùng chuỗi: `(` d=1; `(` d=2; `)` d=1, trước là `(` nên cộng `2^1 = 2`; `(` d=2; `(` d=3; `)` d=2, cộng `2^2 = 4`; `)` d=1; `)` d=0. Tổng `2 + 4 = 6` ✓.

In [14]:
def trace_depth(s):
    res = depth = 0
    print(f"s = {s!r}")
    print(f"{'i':>2} {'ch':>3} {'depth':>6} {'cộng':>6} {'res':>5}")
    for i, ch in enumerate(s):
        add = 0
        if ch == "(":
            depth += 1
        else:
            depth -= 1
            if s[i - 1] == "(":
                add = 1 << depth
                res += add
        print(f"{i:>2} {ch:>3} {depth:>6} {add:>6} {res:>5}")
    print("=>", res)
    return res


assert trace_depth("(()(()))") == 6

s = '(()(()))'
 i  ch  depth   cộng   res
 0   (      1      0     0
 1   (      2      0     0
 2   )      1      2     2
 3   (      2      0     2
 4   (      3      0     2
 5   )      2      4     6
 6   )      1      0     6
 7   )      0      0     6
=> 6


## 9. Code nộp LeetCode

Chọn **đếm độ sâu** (`O(n)` thời gian, `O(1)` bộ nhớ). Giữ nguyên `class Solution`, `scoreOfParentheses(self, s)`:

In [15]:
from typing import List


class Solution:
    def scoreOfParentheses(self, s: str) -> int:
        res = depth = 0
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            else:
                depth -= 1
                if s[i - 1] == "(":                       # vừa đóng một cặp "()" lá
                    res += 1 << depth                     # nhân 2 cho mỗi cặp bao ngoài
        return res


print([Solution().scoreOfParentheses(s) for s, _ in examples])      # [1, 2, 2]
assert all(Solution().scoreOfParentheses(s) == want for s, want in examples)

[1, 2, 2]


## 10. Vì sao đúng

**(a) Điểm là tổng đóng góp của các lá.** Quy nạp theo cấu trúc: `"()"` có đúng một lá ở độ sâu `0`, đóng góp `2^0 = 1` ✓. `AB`: lá của `A` và `B` giữ nguyên độ sâu trong chuỗi ghép, điểm `A + B` là tổng ✓. `(A)`: mỗi lá của `A` tăng độ sâu thêm `1`, đóng góp nhân `2`, nên điểm `2 * A` = tổng đóng góp mới ✓.

**(b) Cách 3 đếm đúng lá và độ sâu.** Lá `"()"` là cặp `'(' ')'` liền nhau: nhận ra bằng `s[i-1] == '('` khi gặp `')'`. Sau khi giảm `depth` ở `')'`, `depth` bằng số `'('` chưa đóng, tức số cặp bao ngoài lá đó.

**(c) Cách 2.** Bất biến: đỉnh stack là điểm đã tích luỹ của cặp đang mở trong cùng (đáy là cấp ngoài cùng). Khi đóng, điểm cặp là `1` nếu bên trong chưa tích luỹ gì (`inner = 0`, vì mọi cặp bên trong có điểm `≥ 1`, nên `inner = 0` đúng khi rỗng) hoặc `2 * inner`, rồi cộng cho cha — khớp luật `AB` (cộng) và `(A)` (nhân đôi).

**(d) Cách 1 và 4** là chuyển nguyên văn định nghĩa thành mã.

Ta kiểm chứng bằng máy: mọi cách khớp **điểm sinh từ chính văn phạm của đề** trên **mọi** chuỗi cân bằng tới 10 cặp (mục 12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy theo định nghĩa | `O(n²)` | `O(n)` | cắt chuỗi con |
| 2. Stack điểm | `O(n)` | `O(n)` | rõ ràng, dễ mở rộng |
| 3. Đếm độ sâu | `O(n)` | `O(1)` | **đáp án chính** |
| 4. Parser đệ quy | `O(n)` | `O(n)` | bám sát văn phạm |

Đo thực tế trên chuỗi lồng sâu và chuỗi phẳng (để thấy chênh lệch ở `n` lớn hơn giới hạn đề):

In [16]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


deep = "(" * 400 + ")" * 400
flat = "()" * 2000
for name, fn in [("recursive", score_recursive), ("stack", score_stack), ("depth", score_depth), ("parser", score_parser)]:
    print(f"{name:10s} deep(n=800): {bench(fn, deep) * 1e3:7.3f} ms | flat(n=4000): {bench(fn, flat) * 1e3:7.3f} ms")

recursive  deep(n=800):   8.299 ms | flat(n=4000):   0.369 ms
stack      deep(n=800):   0.128 ms | flat(n=4000):   0.457 ms
depth      deep(n=800):   0.077 ms | flat(n=4000):   0.465 ms
parser     deep(n=800):   0.161 ms | flat(n=4000):   0.362 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) vét cạn mọi chuỗi cân bằng tới 10 cặp với oracle **sinh từ văn phạm**, (E) ngẫu nhiên có seed và tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [17]:
def grammar_scores(max_pairs):
    """Oracle: sinh MỌI chuỗi cân bằng bằng ba luật của đề, ghi điểm theo luật. Kiểm tra không mâu thuẫn (điểm duy nhất)."""
    by_n = {1: {"()": 1}}
    for n in range(2, max_pairs + 1):
        cur = {}

        def put(string, sc):
            assert cur.get(string, sc) == sc, ("điểm mâu thuẫn", string)
            cur[string] = sc

        for a, sa in by_n[n - 1].items():                      # luật (A) = 2 * A
            put("(" + a + ")", 2 * sa)
        for k in range(1, n):                                  # luật AB = A + B
            for a, sa in by_n[k].items():
                for b, sb in by_n[n - k].items():
                    put(a + b, sa + sb)
        by_n[n] = cur
    return by_n


IMPLS = {
    "recursive": score_recursive,
    "stack": score_stack,
    "depth": score_depth,
    "parser": score_parser,
    "Solution": lambda s: Solution().scoreOfParentheses(s),
    "SolutionStack": lambda s: SolutionStack().scoreOfParentheses(s),
    "SolutionDepth": lambda s: SolutionDepth().scoreOfParentheses(s),
    "SolutionParser": lambda s: SolutionParser().scoreOfParentheses(s),
}


def check(s, want=None):
    for name, fn in IMPLS.items():
        got = fn(s)
        if want is not None:
            assert got == want, (name, s, got, want)


for s, want in examples:
    check(s, want)
G = grammar_scores(3)
assert G[1] == {"()": 1} and G[2] == {"(())": 2, "()()": 2}
print("ví dụ đề: cả", len(IMPLS), "cài đặt đúng; văn phạm sinh n=2:", G[2])

ví dụ đề: cả 8 cài đặt đúng; văn phạm sinh n=2: {'(())': 2, '()()': 2}


### B. Edge case

- Cặp đơn `"()"`.
- Lồng sâu `"((()))"` = `4`.
- Phẳng `"()()()"` = `3`.
- Lồng + phẳng: `"(()())"` = `2·(1+1) = 4`; `"(())()"` = `3`; `"()(())"` = `3`.
- Lồng sâu tối đa theo ràng buộc: `"(" * 25 + ")" * 25` = `2^24`.
- Phẳng dài nhất: `"()" * 25` = `25`.

In [18]:
cases = [
    ("()", 1), ("((()))", 4), ("()()()", 3), ("(()())", 4), ("(())()", 3), ("()(())", 3),
    ("(" * 25 + ")" * 25, 2 ** 24), ("()" * 25, 25), ("(()(()))", 6), ("((())())", 6),
]
for s, want in cases:
    check(s, want)
print(len(cases), "edge case: tất cả cài đặt đúng")

10 edge case: tất cả cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_all_ones(s):                                   # mỗi "()" = 1, bỏ qua nhân đôi
    return sum(1 for i in range(1, len(s)) if s[i] == ")" and s[i - 1] == "(")


def bug_add_not_mul(s):                                # "(A)" tính A + 1 thay vì 2A
    stack = [0]
    for ch in s:
        if ch == "(":
            stack.append(0)
        else:
            inner = stack.pop()
            stack[-1] += inner + 1
    return stack[0]


def bug_depth_off(s):                                  # đếm độ sâu cả chính nó: 2^(d+1)
    res = depth = 0
    for i, ch in enumerate(s):
        if ch == "(":
            depth += 1
        else:
            if s[i - 1] == "(":
                res += 1 << depth                      # SAI: chưa giảm depth
            depth -= 1
    return res


def bug_every_close(s):                                # cộng 2^d cho MỌI ')', không chỉ lá
    res = depth = 0
    for ch in s:
        if ch == "(":
            depth += 1
        else:
            depth -= 1
            res += 1 << depth
    return res


def bug_no_empty_case(s):                              # quên "()" rỗng = 1: dùng 2 * inner luôn
    stack = [0]
    for ch in s:
        if ch == "(":
            stack.append(0)
        else:
            inner = stack.pop()
            stack[-1] += 2 * inner
    return stack[0]


BUGS = [bug_all_ones, bug_add_not_mul, bug_depth_off, bug_every_close, bug_no_empty_case]
G = grammar_scores(6)
allstr = [(s, v) for n in G for s, v in G[n].items()]
for bug in BUGS:
    bad = next(((s, v) for s, v in allstr if bug(s) != v), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:18s} sai trên {bad[0]!r}: ra {bug(bad[0])}, đúng {bad[1]}")

bug_all_ones       sai trên '(())': ra 1, đúng 2
bug_add_not_mul    sai trên '((()))': ra 3, đúng 4
bug_depth_off      sai trên '()': ra 2, đúng 1
bug_every_close    sai trên '(())': ra 3, đúng 2
bug_no_empty_case  sai trên '()': ra 0, đúng 1


### D. Vét cạn mọi chuỗi cân bằng nhỏ

Oracle sinh chuỗi từ ba luật của đề, tới `10` cặp ngoặc. Số chuỗi theo `n` là số Catalan `C_n`; ta kiểm tra luôn số lượng: `1, 2, 5, 14, 42, 132, 429, 1430, 4862, 16796` (tổng `23 714`).

In [20]:
t0 = time.perf_counter()
G = grammar_scores(10)
catalan = [1, 2, 5, 14, 42, 132, 429, 1430, 4862, 16796]
total = 0
for n in range(1, 11):
    assert len(G[n]) == catalan[n - 1], (n, len(G[n]))
    for s, want in G[n].items():
        for name, fn in IMPLS.items():
            assert fn(s) == want, (name, s)
        total += 1
print(total, "chuỗi cân bằng (số lượng khớp Catalan): 8 cài đặt khớp oracle văn phạm;", round(time.perf_counter() - t0, 1), "s")
assert total == sum(catalan)

23713 chuỗi cân bằng (số lượng khớp Catalan): 8 cài đặt khớp oracle văn phạm; 0.5 s


### E. Ngẫu nhiên có seed và tính chất

Sinh chuỗi cân bằng ngẫu nhiên tới 25 cặp (ràng buộc đề) và kiểm tra: mọi cách khớp nhau; thêm hai **tính chất**: (1) `score("(" + s + ")") = 2·score(s)`; (2) `score(a + b) = score(a) + score(b)`.

In [21]:
def random_balanced(rng, pairs):
    s, opened, closed = [], 0, 0
    while closed < pairs:
        if opened < pairs and (opened == closed or rng.random() < 0.5):
            s.append("("); opened += 1
        else:
            s.append(")"); closed += 1
    return "".join(s)


rng = random.Random(20240856)
for _ in range(3000):
    s = random_balanced(rng, rng.randint(1, 25))
    ref = score_depth(s)
    for name, fn in IMPLS.items():
        assert fn(s) == ref, (name, s)
    assert Solution().scoreOfParentheses("(" + s + ")") == 2 * ref
    t = random_balanced(rng, rng.randint(1, 10))
    assert Solution().scoreOfParentheses(s + t) == ref + score_depth(t)
print("3000 chuỗi ngẫu nhiên: 8 cài đặt khớp, tính chất nhân đôi và cộng đều đúng")

3000 chuỗi ngẫu nhiên: 8 cài đặt khớp, tính chất nhân đôi và cộng đều đúng


### F. Giới hạn

Độ dài `50` (25 cặp): điểm lớn nhất là chuỗi lồng sâu `2^24 = 16 777 216`. Đồng thời kiểm tra các chuỗi quá giới hạn đề (n tới 4000) để thấy các cách `O(n)` khớp nhau, và đối chiếu công thức đóng cho vài dạng.

In [22]:
assert Solution().scoreOfParentheses("(" * 25 + ")" * 25) == 2 ** 24 == 16777216
assert Solution().scoreOfParentheses("()" * 25) == 25
big = [random_balanced(rng, 2000) for _ in range(20)] + ["(" * 1000 + ")" * 1000, "()" * 2000]
for s in big:
    ref = score_depth(s)
    assert score_stack(s) == ref
    if len(s) <= 1000:                                  # parser/recursive đệ quy sâu: chỉ chạy cỡ vừa
        assert score_parser(s) == ref and score_recursive(s) == ref
print("chuỗi lớn: stack và depth khớp; công thức đóng 2^(k-1) cho k cặp lồng:", [score_depth("(" * k + ")" * k) == 2 ** (k - 1) for k in (1, 2, 5, 25)])

chuỗi lớn: stack và depth khớp; công thức đóng 2^(k-1) cho k cặp lồng: [True, True, True, True]


## 13. Các lỗi người mới hay gặp

### Lỗi 1: coi mỗi `"()"` là `1` rồi cộng hết

Quên rằng cặp ngoặc bao ngoài **nhân đôi**.

In [23]:
print("(()) ->", bug_all_ones("(())"), "| đúng:", Solution().scoreOfParentheses("(())"))
assert bug_all_ones("(())") == 1 and Solution().scoreOfParentheses("(())") == 2

(()) -> 1 | đúng: 2


**Vì sao sai:** `"(())"` có một lá nhưng ở độ sâu 1 nên là `2`. **Đúng:** lá ở độ sâu `d` đóng góp `2^d`.

### Lỗi 2: cộng 1 thay vì nhân 2 khi đóng cặp

`(A)` là `2 * A` chứ không phải `A + 1`.

In [24]:
print("(()()) ->", bug_add_not_mul("(()())"), "| đúng:", Solution().scoreOfParentheses("(()())"))
assert bug_add_not_mul("(()())") != Solution().scoreOfParentheses("(()())")

(()()) -> 3 | đúng: 4


**Vì sao sai:** `"(()())" = 2·(1+1) = 4`, còn cộng 1 cho `3`. **Đúng:** `stack[-1] += max(1, 2 * inner)`.

### Lỗi 3: lệch độ sâu (dùng `2^(d+1)`)

Cộng `1 << depth` **trước** khi giảm `depth` đếm cả chính cặp lá, cho kết quả gấp đôi.

In [25]:
print("() ->", bug_depth_off("()"), "| đúng:", Solution().scoreOfParentheses("()"))
assert bug_depth_off("()") == 2

() -> 2 | đúng: 1


**Vì sao sai:** lá `"()"` ở độ sâu `0` phải cho `1`. **Đúng:** `depth -= 1` **rồi** `res += 1 << depth`.

### Lỗi 4: cộng điểm cho mọi `')'`

Chỉ lá (`"()"` liền nhau) cộng điểm; `')'` đóng cặp lớn thì không.

In [26]:
print("(()) ->", bug_every_close("(())"), "| đúng:", Solution().scoreOfParentheses("(())"))
assert bug_every_close("(())") == 3

(()) -> 3 | đúng: 2


**Vì sao sai:** đóng cặp ngoài cùng đã được tính qua phép nhân ở lá. **Đúng:** kiểm tra `s[i - 1] == '('`.

### Lỗi 5: quên trường hợp cặp rỗng

Dùng `2 * inner` cho mọi cặp thì `"()"` (inner = 0) ra `0`.

In [27]:
print("() ->", bug_no_empty_case("()"), "| đúng:", Solution().scoreOfParentheses("()"))
assert bug_no_empty_case("()") == 0

() -> 0 | đúng: 1


**Vì sao sai:** `"()"` có điểm `1`, không phải `2·0`. **Đúng:** `max(1, 2 * inner)`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chuỗi ngoặc cân bằng + giá trị phụ thuộc cấu trúc lồng nhau" → **stack** hoặc **đếm độ sâu**. Khi luật là "nhân với hằng mỗi cấp lồng" thì tổng các lá `hằng^độ sâu`.

**Mẹo:**

1. Vẽ chuỗi thành cây rồi đếm lá và độ sâu.
2. Luôn tự hỏi "chuỗi rỗng bên trong cho giá trị gì" (điểm `1` chứ không `0`).
3. Cập nhật `depth` đúng thứ tự (giảm trước khi dùng cho lá).
4. Kiểm tra bằng sinh chuỗi từ văn phạm — oracle độc lập.
5. `1 << d` nhanh và rõ hơn `2 ** d`.

**Họ bài cùng pattern:** Valid Parentheses (20), Longest Valid Parentheses (32), Minimum Add to Make Parentheses Valid (921), Decode String (394), Basic Calculator (224), Remove Outermost Parentheses (1021), Maximum Nesting Depth of the Parentheses (1614).

## 15. Biến thể

1. **Hằng khác 2:** `(A)` có điểm `k * A` — lá `"()"` ở độ sâu `d` đóng góp `k^d`.
2. **Điểm khác cho lá:** `"()"` có điểm `c`.
3. **Chỉ độ sâu tối đa (1614).**
4. **Xoá ngoặc ngoài cùng (1021).**
5. **Đếm số chuỗi cân bằng theo điểm** (quy hoạch động).

Cài biến thể 1 và 2, đối chiếu với bản đệ quy trực tiếp tổng quát:

In [28]:
def score_general(s, k=2, leaf=1):
    res = depth = 0
    for i, ch in enumerate(s):
        if ch == "(":
            depth += 1
        else:
            depth -= 1
            if s[i - 1] == "(":
                res += leaf * k ** depth
    return res


def score_general_rec(s, k, leaf):
    if s == "":
        return 0
    total, bal, start = 0, 0, 0
    for i, ch in enumerate(s):
        bal += 1 if ch == "(" else -1
        if bal == 0:
            inner = s[start + 1:i]
            total += leaf if inner == "" else k * score_general_rec(inner, k, leaf)
            start = i + 1
    return total


G = grammar_scores(8)
for n in G:
    for s in G[n]:
        for k, leaf in [(2, 1), (3, 1), (3, 5), (1, 1)]:
            assert score_general(s, k, leaf) == score_general_rec(s, k, leaf), (s, k, leaf)
assert score_general("(())", 3, 1) == 3 and score_general("(())", 3, 5) == 15
print("biến thể (hằng k, điểm lá c): công thức c * k^d khớp đệ quy tổng quát trên mọi chuỗi tới 8 cặp")


def max_depth(s):                                     # 1614
    d = best = 0
    for ch in s:
        d += 1 if ch == "(" else -1
        best = max(best, d)
    return best


def remove_outer(s):                                  # 1021
    out, bal = [], 0
    for ch in s:
        if ch == "(":
            if bal > 0:
                out.append(ch)
            bal += 1
        else:
            bal -= 1
            if bal > 0:
                out.append(ch)
    return "".join(out)


assert max_depth("(()(()))") == 3 and remove_outer("(()())(())") == "()()()"
print("max_depth và remove_outer đúng")

biến thể (hằng k, điểm lá c): công thức c * k^d khớp đệ quy tổng quát trên mọi chuỗi tới 8 cặp
max_depth và remove_outer đúng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy theo định nghĩa | tách khối, đệ quy trên chuỗi con | `O(n²)` | `O(n)` | oracle, giải thích |
| 2. Stack điểm | điểm tích luỹ mỗi cặp đang mở | `O(n)` | `O(n)` | dễ mở rộng luật |
| 3. Đếm độ sâu | lá cộng `2^d` | `O(n)` | `O(1)` | **đáp án chính** |
| 4. Parser đệ quy | văn phạm trực tiếp | `O(n)` | `O(n)` | bám đề, mở rộng văn phạm |

Cả bốn khớp oracle văn phạm trên 23 714 chuỗi cân bằng và hàng nghìn chuỗi ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().scoreOfParentheses("(()())"), Solution().scoreOfParentheses("((()))"), Solution().scoreOfParentheses("()(())"))
print(max_depth("((())())"), remove_outer("()(())"))

4 4 3
3 ()


### Bài 2 (tính tay)

Tính điểm của `s = "((())())()"` bằng cả hai cách: (a) theo cây (b) theo lá `2^d`. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện stack:

In [30]:
def stack_todo(s):
    stack = [0]
    for ch in s:
        if ch == "(":
            pass            # TODO 1: đẩy điểm tích luỹ mới 0
        else:
            pass            # TODO 2: pop inner, cộng max(1, 2*inner) vào đỉnh
    return stack[0]


# Khi làm xong, bỏ comment để kiểm tra:
# assert [stack_todo(s) for s, _ in examples] == [1, 2, 2] and stack_todo("(()(()))") == 6

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` dùng **đếm độ sâu**:

In [31]:
class MySolution:
    def scoreOfParentheses(self, s):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().scoreOfParentheses(s) == w for s, w in examples)

### Bài 5 (mini challenge): đếm chuỗi theo điểm

Viết `count_by_score(n)` trả dict `{điểm: số chuỗi cân bằng n cặp có điểm đó}` bằng quy hoạch động (không sinh từng chuỗi), rồi so với oracle văn phạm với `n ≤ 8`.

In [32]:
def count_by_score(n):
    pass            # TODO


# Khi làm xong:
# assert count_by_score(3) == {4: 2, 3: 3}

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `"(()())"` = `2 * (1 + 1)`; `"((()))"` = `2 * 2 * 1`.
- Mức 2: `"()(())"` = `1 + 2`.
- Mức 3: kết quả `4 4 3`; `max_depth("((())())") = 3`, `remove_outer("()(())") = "()"`.

**Bài 2**

- Mức 1: tách khối nguyên thể: `((())())` và `()`.
- Mức 2: `((())())` = `2 * ( (()) + () ) = 2 * (2 + 1) = 6`; `()` = `1`.
- Mức 3: tổng `7`; theo lá: lá độ sâu 2 → 4, lá độ sâu 1 → 2, lá độ sâu 0 → 1; `4 + 2 + 1 = 7`.

**Bài 3**

- Mức 1: `stack.append(0)`.
- Mức 2: `inner = stack.pop()`.
- Mức 3: `stack[-1] += max(1, 2 * inner)`.

**Bài 4**

- Mức 1: giữ `res` và `depth`.
- Mức 2: gặp `')'` giảm `depth` rồi kiểm tra `s[i-1] == '('`.
- Mức 3: `res += 1 << depth`.

**Bài 5**

- Mức 1: `f(n)` là dict; `f(1) = {1: 1}`.
- Mức 2: hai luật: bao ngoài `(A)` (điểm `2a`) và ghép `AB` (điểm `a + b`, kích thước `k` và `n - k`).
- Mức 3: dùng khối nguyên thể: `f(n)` = tổ hợp các khối `(A)` với kích thước `m` (điểm `2 * score(A)` hoặc `1` nếu `m = 1`) cộng với phần còn lại.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
assert [Solution().scoreOfParentheses(s) for s in ("(()())", "((()))", "()(())")] == [4, 4, 3]
assert max_depth("((())())") == 3 and remove_outer("()(())") == "()"
print("Bài 1 OK")

Bài 1 OK


### Bài 2

`"((())())()"`: khối `((())())` = `6` và `()` = `1`. Tổng `7`. Theo lá: lá trong cùng ở độ sâu 2 (`4`), lá thứ hai ở độ sâu 1 (`2`), lá ngoài cùng ở độ sâu 0 (`1`).

In [34]:
s = "((())())()"
assert Solution().scoreOfParentheses(s) == 7 == grammar_scores(5)[5][s]
assert 4 + 2 + 1 == 7
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def stack_answer(s):
    stack = [0]
    for ch in s:
        if ch == "(":
            stack.append(0)
        else:
            inner = stack.pop()
            stack[-1] += max(1, 2 * inner)
    return stack[0]


G = grammar_scores(8)
assert all(stack_answer(s) == v for n in G for s, v in G[n].items())
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def scoreOfParentheses(self, s):
        res = depth = 0
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            else:
                depth -= 1
                if s[i - 1] == "(":
                    res += 1 << depth
        return res


assert all(MySolutionAnswer().scoreOfParentheses(s) == v for n in G for s, v in G[n].items())
print("Bài 4 OK")

Bài 4 OK


### Bài 5

`f(n)` = dict điểm → số chuỗi. Dùng văn phạm: `S(n) = (S(n-1))` (nhân đôi) hoặc ghép nguyên thể đầu `(P)` có `m` cặp và phần còn lại `n - m` cặp (rỗng nếu `n == m`). Mỗi chuỗi có phân rã nguyên thể duy nhất nên không đếm trùng.

In [37]:
from collections import Counter
from functools import lru_cache


@lru_cache(None)
def count_by_score_answer(n):
    """Trả tuple((điểm, số chuỗi)) cho chuỗi cân bằng n cặp; n = 0 là chuỗi rỗng điểm 0."""
    if n == 0:
        return ((0, 1),)
    res = Counter()
    for m in range(1, n + 1):                           # khối nguyên thể đầu có m cặp
        if m == 1:
            first = Counter({1: 1})
        else:
            first = Counter({2 * sc: c for sc, c in count_by_score_answer(m - 1)})
        rest = dict(count_by_score_answer(n - m))
        for a, ca in first.items():
            for b, cb in rest.items():
                res[a + b] += ca * cb
    return tuple(sorted(res.items()))


for n in range(1, 9):
    expected = Counter(G[n].values())
    assert dict(count_by_score_answer(n)) == dict(expected), n
print("Bài 5 OK: DP đếm khớp oracle văn phạm với n ≤ 8; n = 3:", dict(count_by_score_answer(3)))

Bài 5 OK: DP đếm khớp oracle văn phạm với n ≤ 8; n = 3: {3: 3, 4: 2}


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (ngoặc lồng, mỗi cấp nhân đôi → tổng lá `2^d`):**

```python
class Solution:
    def scoreOfParentheses(self, s: str) -> int:
        res = depth = 0
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            else:
                depth -= 1
                if s[i - 1] == "(":          # lá "()"
                    res += 1 << depth
        return res
```

**Checklist trước khi submit:**

- [ ] Lá `"()"` ở độ sâu `d` cho `2^d` (độ sâu = số cặp **bao ngoài**, không tính chính nó)?
- [ ] Giảm `depth` **trước** khi cộng `1 << depth`?
- [ ] Chỉ cộng ở lá (`s[i-1] == '('`), không cộng mọi `')'`?
- [ ] Cặp rỗng `()` cho `1`, không `0`?
- [ ] Đã thử `"()"` → 1, `"(())"` → 2, `"()()"` → 2, `"(()(()))"` → 6?
- [ ] Đúng tên `class Solution`, `scoreOfParentheses(self, s)`, không in debug.